# 🤖 Binary Classification Template
### A reusable scaffold for supervised classification projects

---

## How to use this template

1. Search for every `# TODO:` comment and fill in the dataset-specific details
2. Run sections top-to-bottom — later sections depend on variables defined earlier
3. All code cells are intentionally left empty — add your own implementation
4. The markdown explanations describe *what to do* and *why*; the code cells are where you *do it*

**This template follows the same structure as:**
- `07-01-machine-learning.ipynb` — ncbirths low birthweight prediction
- `07-02-california_housing-v2_v9999.ipynb` — California housing price classification


## 1. Imports & Configuration

Import all libraries up front. Use the grouping below as a checklist.

**Mandatory for every classification project:**
- `numpy`, `pandas` — data handling
- `matplotlib`, `seaborn` — visualisation
- `sklearn.model_selection` — `train_test_split`, `cross_validate`, `GridSearchCV`
- `sklearn.pipeline` — `Pipeline`
- `sklearn.compose` — `ColumnTransformer`
- `sklearn.preprocessing` — `StandardScaler`, `OneHotEncoder`, `LabelEncoder`
- `sklearn.metrics` — `roc_auc_score`, `f1_score`, `confusion_matrix`, `roc_curve`

**Add as needed:**
- `sklearn.impute.SimpleImputer` — if your data has missing values
- `xgboost.XGBClassifier`, `lightgbm.LGBMClassifier` — gradient boosting
- `shap` — model interpretability
- `scipy.stats.randint`, `uniform` — if using RandomizedSearchCV
- `sklearn.preprocessing.FunctionTransformer` — if applying log transforms
- `ipywidgets` — for interactive threshold sliders

**Configuration to set here:**
- Colour palette constants (`CLR_TP`, `CLR_TN`, `CLR_FP`, `CLR_FN`)
- `sns.set_theme()` and `plt.rcParams` for consistent plot styling
- Random seeds (`RANDOM_STATE = 123`)


In [ ]:
# TODO: Add your imports here


## 2. Load Data & Feature Engineering

### 2.1 Load raw data

Load your dataset and perform initial inspection:
- `df.shape` — how many rows and columns?
- `df.dtypes` — are columns the right type (numeric vs categorical)?
- `df.head()` — do the values look sensible?
- `df.describe()` — summary statistics for numeric columns

> 📌 **Note the unit of analysis:** Is each row a person? A transaction? A district? A time period? This matters for feature interpretation.


In [ ]:
# TODO: Load your dataset
# df = pd.read_csv('../data/YOUR_FILE.csv')


### 2.2 Feature Engineering

Create new features from raw columns if needed. Common patterns:

| Transformation | When to use | Example |
|----------------|-------------|---------|
| **Ratio features** | Raw totals that depend on group size | `rooms / households` |
| **Log transform** | Right-skewed numeric features | `log1p(income)` |
| **Interaction terms** | When two features combine non-linearly | `age × income` |
| **Binning** | Continuous → ordinal categories | `age_group = pd.cut(age, bins)` |
| **Date decomposition** | Datetime columns | `month`, `day_of_week`, `days_since` |

**Binarise the target (if needed):**
If your target is continuous (e.g. house price, salary), convert it to a binary label:
```python
df['target'] = np.where(df['continuous_col'] >= THRESHOLD, 'above', 'below')
df = df.drop(columns=['continuous_col'])  # CRITICAL: drop source to prevent leakage
```

> ⚠️ **Leakage check:** After creating your target, immediately drop any column that was directly used to derive it. Leaving the source column in features is the most common form of data leakage.


In [ ]:
# TODO: Add feature engineering
# e.g. df['ratio_feature'] = df['col_a'] / df['col_b']
# e.g. df['target'] = np.where(df['value_col'] >= THRESHOLD, 'above', 'below')
# e.g. df = df.drop(columns=['value_col'])  # drop source of target!


## 3. Exploratory Data Analysis (EDA)

**Never fit a model on data you haven't looked at.**

Run all four subsections before moving to modelling.

### 3.1 Class distribution

Check for class imbalance. Questions to answer:
- What is the minority class percentage?
- If < 10%: consider `class_weight='balanced'` and prioritise Recall over Accuracy
- If ~ 50/50: standard metrics (Accuracy, F1) are appropriate
- Always use ROC-AUC as a primary metric regardless of balance

> 🎯 A model that always predicts the majority class can achieve high accuracy while learning nothing. Always prefer AUC-ROC / F1 over raw accuracy as your primary metric.


In [ ]:
# TODO: Check class distribution
# df['target'].value_counts(normalize=True)


### 3.2 Missing values

Identify which columns have missing values and how many:
```python
df.isnull().sum()
df.isnull().mean() * 100  # as percentage
```

**Decision:** Drop rows (`dropna()`) or impute (`SimpleImputer`)?
- **Drop rows:** Simple, safe if < 5% of data is missing
- **Impute in pipeline:** Production-appropriate; use `strategy='median'` for numeric, `'most_frequent'` for categorical


In [ ]:
# TODO: Inspect and handle missing values


### 3.3 Feature distributions by target class

For each numeric feature, compare its distribution across target classes:
- Box plots or violin plots — does the distribution differ across classes?
- Features with large separation between class distributions → strong predictors
- Features with overlapping distributions → weaker predictors (but may still help in combination)


In [ ]:
# TODO: Plot feature distributions by class
# e.g. sns.FacetGrid or sns.boxplot per feature


### 3.4 Correlation matrix

Check for multicollinearity among numeric features:
- Use Spearman correlation for skewed features (robust to non-normality)
- Use Pearson for roughly normal features
- Correlation > ±0.8: consider dropping one of the two collinear features

```python
corr = df[numeric_cols].corr(method='spearman')
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
```


In [ ]:
# TODO: Correlation matrix


## 4. Feature Selection & Train/Test Split

### Select features and target

```python
FEATURES = [
    # TODO: list your feature columns
    # Exclude: the target column, ID columns, date columns (unless you've engineered from them),
    #          any column derived from the target (leakage!)
]
TARGET = 'your_target_column'
```

### The golden rule: split BEFORE preprocessing

1. **Split** the data
2. **Fit** preprocessor on training data only
3. **Transform** training data
4. **Transform** test data using training statistics

**Never** fit the preprocessor on the full dataset before splitting — that leaks test statistics into training.

### Parameters

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,       # 80% train, 20% test
    random_state=123,     # reproducible split
    stratify=y            # preserve class ratio in both splits
)
```

`stratify=y` is important whenever your classes are imbalanced — it ensures both splits have the same class ratio.


In [ ]:
# TODO: Define features and target, then split
# FEATURES = [...]
# TARGET = '...'
#
# X = df[FEATURES]
# y = df[TARGET]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=123, stratify=y)


## 5. Preprocessing Pipeline

### Identify feature types

```python
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features   = X_train.select_dtypes(include=[np.number]).columns.tolist()
```

### Design your transformer branches

| Feature type | Typical pipeline | When |
|---|---|---|
| Normal numeric | `SimpleImputer` → `StandardScaler` | Most numeric features |
| Skewed numeric | `SimpleImputer` → `log1p` → `StandardScaler` | Right-skewed (income, prices) |
| Geographic coords | `SimpleImputer` only | Lat/lon — already on consistent scale |
| Categorical (text) | `SimpleImputer` → `OneHotEncoder` | Nominal categories |
| Ordinal categorical | `SimpleImputer` → `OrdinalEncoder` | Categories with natural order |

### Why pipeline instead of manual transforms?

A `Pipeline(preprocessor + model)` guarantees:
- ✅ No leakage — scaler is fitted on training fold only during CV
- ✅ Clean API — single `.fit()` / `.predict()` call
- ✅ Deployment-ready — can `pickle` the entire pipeline

### Key parameters

- `OneHotEncoder(handle_unknown='ignore')` — gracefully handles categories in test not seen in training
- `OneHotEncoder(drop='first')` — avoids dummy variable trap (multicollinearity); not needed for tree models
- `StandardScaler()` — essential for Logistic Regression and KNN; harmless for trees


In [ ]:
# TODO: Build preprocessing pipeline
#
# numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
# cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
#
# preprocessor = ColumnTransformer([
#     ('num', numeric_pipe, numerical_features),
#     ('cat', cat_pipe,     categorical_features),
# ])


## 6. Helper Functions

Define reusable plotting functions here before the modelling sections.

**Recommended helpers (see 07-01 / 07-02 for full implementations):**

- `get_outcomes(y_true, y_pred)` → returns per-sample `'TP'`/`'TN'`/`'FP'`/`'FN'` labels
- `plot_confusion_matrix(ax, y_true, y_pred, title, y_proba)` → colour-coded 2×2 confusion matrix
- `plot_feature_vs_prediction(ax, X_test, y_true, y_pred, title)` → scatter of a key feature vs prediction outcome

> 💡 Define these once and reuse them across all model comparisons — consistent visuals make comparison easier.


In [ ]:
# TODO: Add helper functions


## 7. Model Building and Evaluation

### 7.1 Label Encoding (if needed)

XGBoost and LightGBM require **integer labels** (0, 1), not strings.

```python
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
```

> ⚠️ Remember: `le.transform()` maps alphabetically. `above → 0`, `below → 1` (or whatever your classes are, sorted alphabetically). Keep track of this mapping when interpreting probabilities.

### 7.2 Model Zoo

Define all models to compare. Start with at least one from each family:

```python
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=123),
    'Decision Tree':       DecisionTreeClassifier(random_state=123),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=123, n_jobs=-1),
    'XGBoost':             XGBClassifier(eval_metric='logloss', random_state=123, verbosity=0),
    'LightGBM':            LGBMClassifier(random_state=123, verbose=-1),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
}
```

| Algorithm | Family | Needs scaling? | Good default? |
|-----------|--------|---------------|---------------|
| Logistic Regression | Linear | ✅ Yes | ✅ Strong baseline |
| Decision Tree | Tree | ❌ No | ⚠️ Overfits easily |
| **Random Forest** | Ensemble (Bagging) | ❌ No | ✅ Robust |
| **XGBoost** | Ensemble (Boosting) | ❌ No | ✅ Usually top performer |
| **LightGBM** | Ensemble (Boosting) | ❌ No | ✅ Faster than XGBoost |
| KNN | Instance-based | ✅ Yes | ⚠️ Slow on large data |


In [ ]:
# TODO: Label encoding (if using XGBoost/LightGBM)
# le = LabelEncoder()
# y_train_enc = le.fit_transform(y_train)
# y_test_enc  = le.transform(y_test)

# TODO: Define model zoo
# models = { ... }


### 7.3 Cross-Validation and Model Comparison

**Why cross-validation instead of a single train/val split?**

A single split gives one noisy performance estimate. k=5 fold CV averages 5 estimates, giving:
- More reliable mean performance
- Standard deviation showing stability across folds

**Always wrap your preprocessor + model in a Pipeline for CV:**

```python
for model_name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    y_cv = y_train_enc if model_name in ('XGBoost', 'LightGBM') else y_train
    cv_output = cross_validate(pipeline, X_train, y_cv, cv=5,
                               scoring={'roc_auc': 'roc_auc', 'f1': 'f1_weighted'},
                               n_jobs=-1)
```

> 💡 `n_jobs=-1` uses all CPU cores. For 6 models × 5 folds = 30 fits, this gives a significant speed-up.

**Primary metric:** `roc_auc` — insensitive to class imbalance, evaluates ranking quality across all thresholds.
**Secondary metric:** `f1_weighted` — accounts for class frequency, threshold-dependent (at 0.5).


In [ ]:
# TODO: Cross-validation loop
# results = []
# for model_name, model in models.items():
#     pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
#     y_cv = y_train_enc if model_name in ('XGBoost', 'LightGBM') else y_train
#     cv_output = cross_validate(pipeline, X_train, y_cv, cv=5,
#                                scoring={'roc_auc': 'roc_auc', 'f1': 'f1_weighted'}, n_jobs=-1)
#     results.append({...})
# results_df = pd.DataFrame(results).sort_values('Mean ROC-AUC', ascending=False)


### 7.4 Visualise Model Performance

Plot mean ROC-AUC with error bars (std) to compare models visually.

```python
sns.barplot(data=results_df, x='Mean ROC-AUC', y='Model', palette='viridis')
```

Also consider: a scatter plot of **training time vs AUC** to identify the speed/accuracy frontier.

### 7.5 Confusion Matrices for Top Models

Fit the top 2 models (e.g. Logistic Regression and the best tree model) on the full training set and plot confusion matrices on the test set. This shows error patterns at the default 0.5 threshold.


In [ ]:
# TODO: Visualise CV results
# sns.barplot(...)

# TODO: Confusion matrices for top 2 models


## 8. Hyperparameter Tuning

### Two-stage approach (recommended for large search spaces)

**Stage 1 — RandomizedSearchCV:** Broadly explore a large parameter space by random sampling.

```python
from scipy.stats import randint, uniform

param_dist = {
    'classifier__n_estimators':  randint(100, 600),
    'classifier__max_depth':     randint(3, 10),
    'classifier__learning_rate': uniform(0.01, 0.29),
}

random_search = RandomizedSearchCV(
    estimator=pipeline, param_distributions=param_dist,
    n_iter=40, scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=123),
    n_jobs=-1, random_state=42
)
random_search.fit(X_train, y_train_enc)
```

**Stage 2 — GridSearchCV:** Narrow grid around Stage 1's best result.

```python
best = random_search.best_params_
grid_params = {
    'classifier__n_estimators': [best['...'] - 50, best['...'], best['...'] + 50],
    # ...
}
grid_search = GridSearchCV(estimator=pipeline, param_grid=grid_params,
                           scoring='roc_auc', cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train_enc)
```

### Single-stage approach (simpler, for smaller search spaces)

For fewer than ~3 parameters with small ranges, skip RandomizedSearch and go straight to GridSearch:

```python
param_grid = {
    'classifier__max_depth': [3, 5, 7, 10],
    'classifier__n_estimators': [100, 200, 300],
}
grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid,
                           scoring='roc_auc', cv=5, n_jobs=-1)
```

> ⚠️ `scoring='roc_auc'` is almost always the right choice for binary classification. Using `accuracy` as the tuning metric will optimise for the majority class.


In [ ]:
# TODO: Hyperparameter tuning
# Choose: single-stage GridSearchCV or two-stage RandomizedSearch → GridSearch
#
# Stage 1 (if using two-stage):
# random_search = RandomizedSearchCV(...)
# random_search.fit(X_train, y_train_enc)
#
# Stage 2:
# grid_search = GridSearchCV(...)
# grid_search.fit(X_train, y_train_enc)


## 9. Final Model Evaluation on Test Set

### The moment of truth

Everything above — CV, tuning — was done on training data. **The test set has not been touched.**

Now evaluate the best model on held-out test data:

```python
best_pipeline = grid_search.best_estimator_
best_pipeline.fit(X_train, y_train_enc)   # refit on full training set

y_pred       = best_pipeline.predict(X_test)
y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]
```

### Six metrics to report

| Metric | When it matters most |
|--------|---------------------|
| **Accuracy** | Balanced classes only |
| **Precision** | Cost of false alarms is high (e.g. spam detection) |
| **Recall** | Cost of missing positives is high (e.g. cancer screening) |
| **F1-Score** | Need to balance precision and recall |
| **ROC-AUC** | Always — threshold-agnostic ranking metric |
| **Cohen's Kappa** | Comparing models on different datasets; chance-corrected |

> 💡 A gap between CV score and test score is normal. A *large* gap (> 5 points AUC) suggests overfitting — consider more regularisation or simpler model.

### 9.1 Confusion Matrix and Scatter Plot

```python
plot_confusion_matrix(ax, y_test_enc, y_pred, title, y_proba=y_pred_proba)
plot_feature_vs_prediction(ax, X_test, y_test_enc, y_pred, title)
```



In [ ]:
# TODO: Final evaluation
# best_pipeline = grid_search.best_estimator_
# best_pipeline.fit(X_train, y_train_enc)
# y_pred       = best_pipeline.predict(X_test)
# y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]
#
# Report metrics:
# accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
#
# Confusion matrix:
# plot_confusion_matrix(...)



## 10. Feature Importance

### Gain-based importance (tree models)

```python
final_classifier = best_pipeline.named_steps['classifier']
importances = final_classifier.feature_importances_
```

### Coefficient importance (Logistic Regression)

```python
importances = np.abs(final_classifier.coef_[0])
```

### Recover clean feature names after preprocessing

After `ColumnTransformer`, feature names get pipeline prefixes (`log__`, `passthrough__`, `cat__`).
Recover them like this:

```python
# For OneHotEncoded columns:
ohe_cats = (fitted_preprocessor
            .named_transformers_['cat']
            .named_steps['ohe']
            .get_feature_names_out(CAT_FEATURES))
feature_names = np.array(NUMERIC_FEATURES + list(ohe_cats))
```

> ⚠️ **Caveat:** Feature importance shows *which features matter*, not *in which direction*. Use SHAP for directional interpretation.


In [ ]:
# TODO: Feature importance
# feature_names = np.array([...])  # reconstruct after pipeline
# importances = best_pipeline.named_steps['classifier'].feature_importances_
# importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values('Importance', ascending=False)
# sns.barplot(data=importance_df.head(10), x='Importance', y='Feature')


## 11. SHAP Values for Model Interpretability

SHAP (SHapley Additive exPlanations) explains *why* the model made each prediction.

### Setup

```python
X_test_transformed  = best_pipeline.named_steps['preprocessor'].transform(X_test)
X_test_display      = pd.DataFrame(X_test_transformed, columns=clean_names)
X_train_transformed = best_pipeline.named_steps['preprocessor'].transform(X_train)
background          = shap.sample(pd.DataFrame(X_train_transformed, columns=clean_names), 100, random_state=42)

explainer   = shap.Explainer(best_pipeline.named_steps['classifier'], background)
shap_values = explainer(X_test_display)

# For binary classification, slice class 1:
if len(shap_values.shape) == 3:
    values_to_plot = shap_values[:, :, 1]
else:
    values_to_plot = shap_values
```

### Three plots — use all three

| # | Plot | Code | Question answered |
|---|------|------|-------------------|
| 1 | **Beeswarm** | `shap.plots.beeswarm(values_to_plot)` | Which features matter globally, and in which direction? |
| 2 | **Waterfall** | `shap.plots.waterfall(shap_values[i])` | How was this one prediction built step by step? |
| 3 | **Dependence** | `shap.dependence_plot(feature, values, X_display)` | How does one feature's effect vary across its range? |

### Force plot (optional)

```python
shap.force_plot(base_value, shap_vals, X_test_display.iloc[i], matplotlib=True)
```
The Force Plot is an alternative to the Waterfall for a single prediction — redundant if you already have the Waterfall.

### Reading SHAP values

- **Positive SHAP** → feature pushed prediction toward class 1
- **Negative SHAP** → feature pushed prediction toward class 0
- `E[f(X)]` = baseline (average prediction across training set)
- `f(x)` = final prediction for this specific sample = baseline + sum of all SHAP values


In [ ]:
# TODO: SHAP explainer
# explainer = shap.Explainer(trained_classifier, background)
# shap_values = explainer(X_test_display)
# values_to_plot = shap_values[:, :, 1] if len(shap_values.shape) == 3 else shap_values


In [ ]:
# TODO: SHAP Plot 1 — Beeswarm (global importance + direction)
# shap.plots.beeswarm(values_to_plot, max_display=12)


In [ ]:
# TODO: SHAP Plot 2 — Waterfall (single prediction)
# sample_idx = 0
# shap.plots.waterfall(shap_values[sample_idx], max_display=12)


In [ ]:
# TODO: SHAP Plot 3 — Dependence (feature effect across its range)
# top_feature = shap_importance_df['Feature'].iloc[0]
# shap.dependence_plot(top_feature, plot_values, X_test_display, interaction_index='auto')


## 12. Key Insights from SHAP Analysis

Summarise the story told by the three SHAP plots in plain English:

1. **Which feature dominates?** (Top of the Beeswarm)
2. **What direction does it push?** (Red dots left or right)
3. **Is there a non-linear threshold?** (Shape of the Dependence Plot)
4. **What was the key driver for Sample #0?** (Waterfall Plot)
5. **What is the population baseline probability?** (`E[f(X)]` from Waterfall)

Use a table like this to summarise:

| Feature | Direction | Effect |
|---------|-----------|--------|
| `feature_1` | High values → ⬇️ probability | Protective |
| `feature_2` | High values → ⬆️ probability | Risk factor |
| ... | | |


In [ ]:
# TODO: Print SHAP key insights summary
# Use shap_importance_df to list top features
# Describe directional effects from beeswarm
# Describe individual case from waterfall


## 13. Conclusion

### Summary checklist

Replace each item with a ✅ as you complete it:

- ⬜ Loaded and explored the dataset
- ⬜ Engineered features (if applicable)
- ⬜ Split data (stratified 80/20)
- ⬜ Built preprocessing pipeline (impute → transform → scale → encode)
- ⬜ Compared multiple classifiers using 5-fold cross-validation
- ⬜ Tuned hyperparameters (RandomizedSearch → GridSearch or GridSearch only)
- ⬜ Evaluated final model on hold-out test set (6 metrics)
- ⬜ Plotted confusion matrix and ROC curve
- ⬜ Analysed feature importance
- ⬜ Explained predictions using SHAP (Beeswarm + Waterfall + Dependence)

### Common pitfalls to check before submitting

| Pitfall | How to check |
|---------|--------------|
| Data leakage | Did you fit any preprocessor before splitting? |
| Wrong label encoding | Are XGBoost/LightGBM using integer labels? |
| Test set contamination | Did you ever use `y_test` to make modelling decisions? |
| Reporting CV score as final score | Did you evaluate the final model on held-out test data? |
| Imbalanced metric choice | If classes are imbalanced, are you using AUC/F1 rather than accuracy? |

### What to learn next

- **More algorithms:** SVM, Naive Bayes, Neural Networks (MLPClassifier)
- **Class imbalance:** SMOTE oversampling, `class_weight='balanced'`
- **Feature selection:** RFECV (recursive feature elimination with CV)
- **Ensemble stacking:** Use predictions of base models as features for a meta-model
- **Deployment:** `joblib.dump(best_pipeline, 'model.pkl')` → load and serve via API
- **Calibration:** `CalibratedClassifierCV` if predicted probabilities need to be well-calibrated


In [ ]:
# TODO: Final performance archive
# summary_df = pd.DataFrame(list(final_metrics.items()), columns=['Metric', 'Score'])
# print(summary_df.to_string(index=False))
